# 07 Atmospheric Turbulence and Closure Phase

**Series:** Physics of the CHARA Array (Mount Wilson Observatory)
**Topic T7:** what turbulent air does to starlight, and the trick that recovers phase information anyway.

Notebooks 03 to 05 assumed each baseline measures a complex visibility: a contrast **and** a phase (the fringe position). The atmosphere ruins the phase. Pockets of warmer and cooler air bend and delay the light, differently above each telescope, and change many times a second.

This notebook covers:

1. Turbulence as a bumpy wavefront, and what it does to one telescope's image.
2. **Piston**: the random extra delay above each telescope, which moves the fringes.
3. Why exposures must be short.
4. **Closure phase**: adding phases around a triangle of telescopes cancels the atmosphere.
5. Real closure phases from CHARA's MIRC-X instrument on the red giant CL Lac.

In [ ]:
%matplotlib inline
import csv
import io
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.optimize import curve_fit
from scipy.special import j1
from ipywidgets import interact, FloatSlider, FloatLogSlider, Dropdown

RAD_TO_MAS = 180 / np.pi * 3600 * 1000
MAS_TO_RAD = 1 / RAD_TO_MAS

BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8a85"
SLIDER_STYLE = {"handle_color": BLUE}  # visible knob on white backgrounds
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "lines.linewidth": 2,
})

STATIONS = {
    "S1": (0.000, 0.000, 0.000),
    "S2": (-5.748, 33.581, 0.637),
    "E1": (125.333, 305.928, -5.919),
    "E2": (70.389, 269.715, -2.803),
    "W1": (-175.068, 216.327, -10.798),
    "W2": (-69.085, 199.342, 0.471),
}
LATITUDE = np.radians(34.2244)
MIRROR = 1.0         # CHARA telescope diameter, m
OUTER_SCALE = 25.0   # largest turbulent eddies, m (a typical value)
R0_WAVELENGTH = 0.5  # r0 is quoted at 0.5 µm


def project(enu, hour_angle, dec):
    # Projected baseline (u, v) in metres, as in notebook 05
    e, n, up = enu
    x = -np.sin(LATITUDE) * n + np.cos(LATITUDE) * up
    z = np.cos(LATITUDE) * n + np.sin(LATITUDE) * up
    u = np.sin(hour_angle) * x + np.cos(hour_angle) * e
    v = (-np.sin(dec) * np.cos(hour_angle) * x + np.sin(dec) * np.sin(hour_angle) * e
         + np.cos(dec) * z)
    return u, v


def wrap(degrees):
    return (degrees + 180) % 360 - 180

## 1. Turbulence as a bumpy wavefront

Starlight arrives as a flat wavefront. Turbulent air delays different parts of it by different amounts, so it arrives bumpy. The size of the bumps is described by one number, the **Fried parameter** $r_0$: the patch size over which the wavefront is still roughly flat. At a good site $r_0$ is about 10 cm for visible light.

$r_0$ grows with wavelength as $\lambda^{6/5}$. The air delays all colours by nearly the same distance, and a fixed delay is a smaller fraction of a long wave. In the H band (1.65 µm) $r_0$ is about four times larger than at 0.5 µm.

The number that matters is $D / r_0$, how many flat patches fit across the mirror.

- $D / r_0 \lesssim 1$: the image is close to the perfect diffraction pattern from notebook 01, but it jitters.
- $D / r_0 \gg 1$: the image breaks into many **speckles**, and averaged over time it blurs to about $\lambda / r_0$ instead of $\lambda / D$.

The simulated wavefront below uses the standard Kolmogorov statistics of turbulence, with a 25 m outer scale (the largest eddies). The pattern of turbulence is fixed, so moving the sliders shows the same air under different conditions.

In [ ]:
SCREEN_N, SCREEN_DX = 256, MIRROR / 64   # 4 m of air, the mirror spans 64 samples
PAD = 512


def phase_screen(n, dx, r0, rng):
    # Kolmogorov / von Kármán phase screen in radians at R0_WAVELENGTH (FFT method)
    f = np.fft.fftfreq(n, dx)
    fx, fy = np.meshgrid(f, f)
    psd = 0.023 * r0 ** (-5 / 3) * (fx ** 2 + fy ** 2 + 1 / OUTER_SCALE ** 2) ** (-11 / 6)
    psd[0, 0] = 0
    noise = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))
    return np.real(np.fft.ifft2(noise * np.sqrt(psd) / (n * dx))) * n * n


UNIT_SCREEN = phase_screen(SCREEN_N, SCREEN_DX, 1.0, np.random.default_rng(7))  # r0 = 1 m
axis_m = (np.arange(SCREEN_N) - SCREEN_N / 2) * SCREEN_DX
px, py = np.meshgrid(axis_m, axis_m)
PUPIL = np.hypot(px, py) <= MIRROR / 2

WAVELENGTHS = {"V, 0.55 µm (visible)": 0.55, "R, 0.70 µm (VEGA)": 0.70,
               "H, 1.65 µm (MIRC-X)": 1.65, "K, 2.2 µm": 2.2}


def star_image(phase):
    field = np.zeros((PAD, PAD), complex)
    c = PAD // 2 - SCREEN_N // 2
    field[c:c + SCREEN_N, c:c + SCREEN_N] = PUPIL * np.exp(1j * phase)
    image = np.abs(np.fft.fftshift(np.fft.fft2(field))) ** 2
    return image / image.max()


@interact(
    r0_cm=FloatSlider(value=10, min=4, max=20, step=1, description="r0 (cm)",
                      continuous_update=False, style=SLIDER_STYLE),
    band=Dropdown(options=list(WAVELENGTHS), value="H, 1.65 µm (MIRC-X)", description="Band"),
)
def show_turbulence(r0_cm, band):
    wavelength = WAVELENGTHS[band]
    r0 = r0_cm / 100
    phase_500 = UNIT_SCREEN * r0 ** (-5 / 6)               # radians at 0.5 µm
    phase = phase_500 * R0_WAVELENGTH / wavelength          # same delay, longer wave
    waves = phase / (2 * np.pi)
    r0_here = r0 * (wavelength / R0_WAVELENGTH) ** 1.2

    pix_mas = wavelength * 1e-6 / (PAD * SCREEN_DX) * RAD_TO_MAS
    half = int(np.clip(12 * MIRROR / r0_here, 80, 250))  # show the whole speckle halo
    crop = slice(PAD // 2 - half, PAD // 2 + half)
    extent = [-half * pix_mas, half * pix_mas] * 2
    seen = star_image(phase)[crop, crop]
    perfect = star_image(np.zeros_like(phase))[crop, crop]

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
    lim = np.abs(waves).max()
    im = axes[0].imshow(waves, extent=[axis_m[0], axis_m[-1]] * 2, origin="lower", cmap="RdBu_r",
                        vmin=-lim, vmax=lim)
    axes[0].add_patch(plt.Circle((0, 0), MIRROR / 2, fill=False, color="black", lw=1.5))
    axes[0].set_title("Wavefront delay over 4 m of air (waves)", fontsize=10)
    axes[0].set_xlabel("m")
    fig.colorbar(im, ax=axes[0], fraction=0.046)
    for ax, img, title in [(axes[1], seen, "Star through the 1 m mirror, one instant"),
                           (axes[2], perfect, "Same mirror, no atmosphere")]:
        ax.imshow(img ** 0.5, extent=extent, origin="lower", cmap="inferno")
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("mas")
    for ax in axes:
        ax.grid(False)
    plt.tight_layout()
    plt.show()
    print(f"r0 at {wavelength} µm: {100 * r0_here:.0f} cm, so D/r0 = {MIRROR / r0_here:.1f}")
    print(f"Perfect image core λ/D = {wavelength * 1e-6 / MIRROR * RAD_TO_MAS:.0f} mas, "
          f"turbulence blur λ/r0 ≈ {wavelength * 1e-6 / r0_here * RAD_TO_MAS:.0f} mas")

**Try this:**

- In the visible (V band), the image breaks into many speckles.
- Switch to H band. $D / r_0$ is about 2, so the image is a slightly distorted version of the perfect one. This is one reason MIRC-X works in the infrared.
- Lower $r_0$ to 4 cm (poor seeing). Even H band starts to break up.

CHARA's telescopes have since been fitted with adaptive optics that flatten most of these bumps across each mirror. Adaptive optics cannot fix the next problem, because it works one telescope at a time.

## 2. Piston: a random delay above each telescope

Average the bumpy wavefront over the whole mirror. What is left is a single number per telescope: how much, on average, that telescope's light was delayed. This is called **piston**.

For interferometry, piston is what matters. The difference in piston between two telescopes is an extra path difference, exactly like the geometric delay in notebook 06, but random. It shifts the fringes sideways.

The turbulence drifts across the telescopes with the wind (the **frozen flow** picture), so each telescope's piston changes with time. CHARA's telescopes are 34 to 331 m apart, much farther than the 25 m outer scale, so this simulation treats the air above each telescope as independent.

In [ ]:
def unit_piston_psd():
    # Spatial power spectrum of mirror-averaged phase along the wind direction, for r0 = 1 m.
    # Computed once on a log grid, then interpolated.
    k = np.geomspace(1e-4, 200, 400)                  # cycles per metre along the wind
    ky = np.concatenate([[0], np.geomspace(1e-5, 60, 1500)])
    kk = np.hypot(k[:, None], ky[None, :])
    turbulence = 0.023 * (kk ** 2 + 1 / OUTER_SCALE ** 2) ** (-11 / 6)
    x = np.pi * MIRROR * np.maximum(kk, 1e-12)
    mirror_average = (2 * j1(x) / x) ** 2
    return k, 2 * np.trapezoid(turbulence * mirror_average, ky, axis=1)


PSD_K, PSD_UNIT = unit_piston_psd()


def piston_opd_um(n_tel, duration_s, dt_s, r0, wind, rng):
    # Random piston delay for each telescope, in micrometres of optical path
    n = int(round(duration_s / dt_s))
    step = wind * dt_s                                 # metres of air passing per sample
    k = np.abs(np.fft.fftfreq(n, step))
    psd = np.exp(np.interp(np.log(np.maximum(k, PSD_K[0])), np.log(PSD_K), np.log(PSD_UNIT)))
    psd *= r0 ** (-5 / 3)
    amplitude = np.sqrt(psd / (n * step))
    noise = rng.normal(size=(n_tel, n)) + 1j * rng.normal(size=(n_tel, n))
    phase = np.real(np.fft.ifft(noise * amplitude, axis=1)) * n   # radians at 0.5 µm
    return phase * R0_WAVELENGTH / (2 * np.pi)


DT = 1e-3       # 1 ms samples
H_BAND = 1.65   # µm


@interact(
    r0_cm=FloatSlider(value=10, min=4, max=20, step=1, description="r0 (cm)",
                      continuous_update=False, style=SLIDER_STYLE),
    wind=FloatSlider(value=10, min=2, max=30, step=1, description="wind (m/s)",
                     continuous_update=False, style=SLIDER_STYLE),
)
def show_piston(r0_cm, wind):
    rng = np.random.default_rng(1)
    opd = piston_opd_um(3, 10, DT, r0_cm / 100, wind, rng)
    t = np.arange(opd.shape[1]) * DT
    difference = opd[0] - opd[1]
    fringe_phase = wrap(360 * difference / H_BAND)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for row, name, colour in zip(opd, ["S1", "E1", "W1"], [BLUE, ORANGE, AQUA]):
        ax1.plot(t, row, color=colour, lw=1.2, label=name)
    ax1.set_ylabel("Piston delay (µm)")
    ax1.set_title(f"Random delay above each telescope, r0 = {r0_cm:.0f} cm, wind {wind:.0f} m/s",
                  fontsize=10)
    ax1.legend(fontsize=8, loc="upper right", ncol=3)
    ax2.scatter(t, fringe_phase, s=1, color=BLUE)
    ax2.set_ylim(-185, 185)
    ax2.set_yticks([-180, -90, 0, 90, 180])
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Fringe phase (°)")
    ax2.set_title("S1-E1 fringe position in H band: scrambled", fontsize=10)
    plt.tight_layout()
    plt.show()

    # Time for the S1-E1 fringe phase to change by 1 radian rms
    lags = np.arange(1, 400)
    change = [np.std(difference[lag:] - difference[:-lag]) * 2 * np.pi / H_BAND for lag in lags]
    tau = lags[np.searchsorted(change, 1.0)] * DT if change[-1] > 1 else np.nan
    print(f"S1-E1 path difference: {difference.std():.1f} µm rms "
          f"({difference.std() / H_BAND:.1f} H-band wavelengths)")
    print(f"The fringe phase changes by 1 radian in about {1000 * tau:.0f} ms")

**Try this:**

- At the defaults the S1-E1 path difference wanders by a few micrometres, a few wavelengths. The fringe phase is completely scrambled within a second.
- Raise the wind to 30 m/s. The pattern is the same but plays faster, so the phase changes in a few milliseconds.
- Raise $r_0$ to 20 cm. The wandering shrinks, but the phase is still scrambled.

A few micrometres also moves the whole fringe packet. That is why the fringe tracker in notebook 06 must keep correcting the carts.

## 3. Short exposures

A camera exposure adds up the fringe over its length. If the fringe moves during the exposure, bright and dark stripes blur together and the contrast drops.

The picture below treats each millisecond's fringe as an arrow on a circle: its length is the contrast, its direction the fringe phase. The exposure measures the **average** arrow (black).

- Short exposure: the arrows point the same way and the average keeps its full length.
- Long exposure: the arrows spread around the circle and the average shrinks.

So MIRC-X takes frames a few milliseconds long. Each frame's phase is random, so frames cannot be averaged as arrows. Instead it averages the **squared** contrast, $V^2$, which does not depend on phase. That is why notebook 04 fitted $V^2$.

In [ ]:
PHASOR_OPD = piston_opd_um(2, 20, DT, 0.10, 10, np.random.default_rng(2))
PHASOR = np.exp(2j * np.pi * (PHASOR_OPD[0] - PHASOR_OPD[1]) / H_BAND)


def contrast_kept(exposure_ms):
    n = max(int(round(exposure_ms)), 1)
    usable = len(PHASOR) // n * n
    return np.abs(PHASOR[:usable].reshape(-1, n).mean(axis=1)).mean()


@interact(exposure_ms=FloatLogSlider(value=10, base=10, min=0, max=3, step=0.1,
                                     description="exposure (ms)", continuous_update=False,
                                     style=SLIDER_STYLE))
def show_exposure(exposure_ms):
    n = max(int(round(exposure_ms)), 1)
    chunk = PHASOR[5000:5000 + n]
    mean = chunk.mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [1, 1.4]})
    circle = np.exp(1j * np.linspace(0, 2 * np.pi, 200))
    ax1.plot(circle.real, circle.imag, color="#c3c2b7", lw=1)
    ax1.scatter(chunk.real, chunk.imag, c=np.arange(n), cmap="viridis", s=12, zorder=3)
    ax1.annotate("", xy=(mean.real, mean.imag), xytext=(0, 0),
                 arrowprops=dict(arrowstyle="->", color="black", lw=2.5))
    ax1.set_aspect("equal")
    ax1.set_xlim(-1.2, 1.2)
    ax1.set_ylim(-1.2, 1.2)
    ax1.set_title(f"One {exposure_ms:.0f} ms exposure: each dot is 1 ms\n"
                  f"average arrow length {abs(mean):.2f}", fontsize=10)

    exposures = np.geomspace(1, 1000, 30)
    ax2.plot(exposures, [contrast_kept(e) for e in exposures], color=BLUE)
    ax2.scatter([exposure_ms], [contrast_kept(exposure_ms)], s=60, color=ORANGE, zorder=3)
    ax2.set_xscale("log")
    ax2.set_ylim(0, 1.05)
    ax2.set_xlabel("Exposure time (ms)")
    ax2.set_ylabel("Fringe contrast kept")
    ax2.set_title("Average over 20 s of simulated turbulence (r0 10 cm, wind 10 m/s)", fontsize=10)
    plt.tight_layout()
    plt.show()

## 4. Closure phase

Each telescope adds its own random delay. Call the piston phases $\epsilon_1, \epsilon_2, \epsilon_3$. The measured fringe phase on each baseline is the true phase plus the difference of two pistons:

$$\phi_{12} = \Phi_{12} + \epsilon_1 - \epsilon_2, \qquad \phi_{23} = \Phi_{23} + \epsilon_2 - \epsilon_3, \qquad \phi_{31} = \Phi_{31} + \epsilon_3 - \epsilon_1$$

Add the three. Every $\epsilon$ appears once with a plus and once with a minus, so they all cancel:

$$\phi_{12} + \phi_{23} + \phi_{31} = \Phi_{12} + \Phi_{23} + \Phi_{31}$$

This sum is the **closure phase**. It ignores the atmosphere entirely and depends only on the star. The idea came from radio astronomy (Jennison, 1958).

What it tells you:

- A **symmetric** source, such as a point, a uniform disk or an equal binary, has a closure phase of exactly 0° or 180°.
- Any other value means the source is **asymmetric**: an unequal binary, a bright spot, a lopsided disk.

Below, three CHARA telescopes (S1, E1, W1) watch a source through the simulated atmosphere.

In [ ]:
SOURCES = {
    "Point star": [(0.0, 0.0, 1.0)],
    "Equal binary, 1.2 mas apart": [(-0.42, -0.42, 1.0), (0.42, 0.42, 1.0)],
    "Unequal binary, 0.8 mas, 2:1": [(-0.267, 0.0, 1.0), (0.533, 0.0, 0.5)],
    "Unequal binary, 0.6 mas, 5:1": [(-0.21, 0.0, 1.0), (0.39, 0.0, 0.2)],
}
TRIANGLE = ["S1", "E1", "W1"]


def source_visibility(points, u_m, v_m, wavelength_um=H_BAND):
    # Complex visibility of a set of point stars (x east, y north in mas, flux)
    u, v = u_m / (wavelength_um * 1e-6), v_m / (wavelength_um * 1e-6)
    total = sum(f * np.exp(-2j * np.pi * (u * x + v * y) * MAS_TO_RAD) for x, y, f in points)
    return total / sum(f for _, _, f in points)


@interact(source=Dropdown(options=list(SOURCES), value="Unequal binary, 0.8 mas, 2:1",
                          description="Source", layout={"width": "360px"}))
def show_closure(source):
    rng = np.random.default_rng(4)
    opd = piston_opd_um(3, 5, DT, 0.10, 10, rng)
    t = np.arange(opd.shape[1]) * DT
    legs = [(0, 1), (1, 2), (2, 0)]
    true_phase, measured = [], []
    for i, j in legs:
        u, v = project(np.subtract(STATIONS[TRIANGLE[j]], STATIONS[TRIANGLE[i]]), 0.0, np.radians(40))
        true = np.degrees(np.angle(source_visibility(SOURCES[source], u, v)))
        true_phase.append(true)
        measured.append(true + 360 * (opd[i] - opd[j]) / H_BAND)
    closure = wrap(sum(measured))
    true_closure = wrap(sum(true_phase))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for row, (i, j), colour in zip(measured, legs, [BLUE, ORANGE, AQUA]):
        ax1.scatter(t, wrap(row), s=1, color=colour, label=f"{TRIANGLE[i]}-{TRIANGLE[j]}")
    ax1.set_ylim(-185, 185)
    ax1.set_yticks([-180, 0, 180])
    ax1.set_ylabel("Phase (°)")
    ax1.set_title("Measured fringe phase on each baseline", fontsize=10)
    ax1.legend(fontsize=8, loc="upper right", markerscale=6, ncol=3)
    ax2.plot(t, closure, color=BLUE)
    ax2.set_ylim(-185, 185)
    ax2.set_yticks([-180, -90, 0, 90, 180])
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Phase (°)")
    ax2.set_title(f"Closure phase (sum of the three): {true_closure:.1f}°", fontsize=10)
    plt.tight_layout()
    plt.show()
    print("True baseline phases: " + ", ".join(f"{p:.1f}°" for p in true_phase))

**Try this:**

- Point star and equal binary: the closure phase is 0° or 180°, whatever the baseline phases are. A symmetric source cannot produce anything else.
- Unequal binaries: the closure phase sits at another steady value, about −83° for the 2:1 pair and −25° for the fainter 5:1 companion. That value is a fingerprint of the asymmetry.

### How much phase information survives

$N$ telescopes have $N(N-1)/2$ baselines, each with an unknown phase. They share only $N - 1$ unknown pistons (one telescope serves as reference). So $(N-1)(N-2)/2$ independent closure phases survive.

| Telescopes | Baseline phases | Independent closure phases | Fraction recovered |
|---|---|---|---|
| 2 | 1 | 0 | 0% |
| 3 | 3 | 1 | 33% |
| 4 | 6 | 3 | 50% |
| 6 | 15 | 10 | 67% |

Combining all six telescopes, as MIRC-X does, recovers two thirds of the phase information. MIRC-X files list all 20 triangles, but only 10 are independent.

## 5. Real closure phases: CL Lac

CL Lac is a pulsating red giant (a Mira variable, spectral type M6) about 1.2 kpc away. In July 2019 Chiavassa et al. observed it with all six CHARA telescopes and MIRC-X (A&A 640, A23, 2020). The calibrated data are public. They contain $V^2$ on 15 baselines and closure phases on 20 triangles, in 8 spectral channels across the H band.

The cell downloads about 4 MB the first time it runs. The data are shared with notebooks 06 and 08. Measurements with fewer than 4 usable spectral channels are skipped, because a few of them are clearly wrong (notebook 08 gives an example).

In [ ]:
TAP_URL = "https://tap.jmmc.fr/vollt/tap/sync"
OIDB_URL = "https://oidb.jmmc.fr"
CL_LAC_DIR = Path("data/irc_50448")


def download_cl_lac():
    # Public MIRC-X data on CL Lac (IRC+50448) from the JMMC Optical Interferometry Database
    done_marker = CL_LAC_DIR / ".complete"
    if done_marker.exists():
        return sorted(CL_LAC_DIR.glob("*.fits"))
    CL_LAC_DIR.mkdir(parents=True, exist_ok=True)
    query = ("SELECT access_url FROM oidb WHERE facility_name = 'CHARA' AND data_rights = 'public' "
             "AND calib_level = 3 AND target_name = 'IRC+50448'")
    params = urllib.parse.urlencode({"REQUEST": "doQuery", "LANG": "ADQL",
                                     "FORMAT": "csv", "QUERY": query}).encode()
    with urllib.request.urlopen(TAP_URL, data=params, timeout=60) as response:
        rows = list(csv.DictReader(io.StringIO(response.read().decode())))

    def fetch(row):
        url = urllib.parse.urljoin(OIDB_URL, row["access_url"])
        target = CL_LAC_DIR / Path(urllib.parse.urlparse(url).path).name
        if not target.exists():
            urllib.request.urlretrieve(url, target)

    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(fetch, rows))
    done_marker.touch()
    return sorted(CL_LAC_DIR.glob("*.fits"))


vis2_sf, vis2, vis2_err = [], [], []
cp_sf, cp, cp_err = [], [], []
for path in download_cl_lac():
    with fits.open(path) as hdul:
        wavelengths = hdul["OI_WAVELENGTH"].data["EFF_WAVE"]
        for row in hdul["OI_VIS2"].data:
            ok = ~row["FLAG"] & (row["VIS2ERR"] > 0)
            if ok.sum() < 4:  # rows with most channels flagged are unreliable
                continue
            vis2_sf.append((np.hypot(row["UCOORD"], row["VCOORD"]) / wavelengths)[ok])
            vis2.append(row["VIS2DATA"][ok])
            vis2_err.append(row["VIS2ERR"][ok])
        for row in hdul["OI_T3"].data:
            ok = ~row["FLAG"] & (row["T3PHIERR"] > 0)
            if ok.sum() < 4:
                continue
            legs = [np.hypot(row["U1COORD"], row["V1COORD"]), np.hypot(row["U2COORD"], row["V2COORD"]),
                    np.hypot(row["U1COORD"] + row["U2COORD"], row["V1COORD"] + row["V2COORD"])]
            cp_sf.append((max(legs) / wavelengths)[ok])
            cp.append(row["T3PHI"][ok])
            cp_err.append(row["T3PHIERR"][ok])
vis2_sf, vis2, vis2_err, cp_sf, cp, cp_err = map(np.concatenate,
                                                 (vis2_sf, vis2, vis2_err, cp_sf, cp, cp_err))
print(f"{len(vis2)} V² points and {len(cp)} closure phases")

First the size. A uniform disk fitted to the first lobe of $V^2$ (spatial frequencies below 40 million wavelengths) gives a rough diameter. Beyond the first null the star is heavily resolved, and $V^2$ falls below 1%.

Then the closure phases, plotted against the longest baseline in each triangle. A symmetric star can only give 0° or ±180°. The right panel measures how far each closure phase is from the nearer of those values, in units of its error bar.

In [ ]:
def uniform_disk_v2(sf, diameter_mas):
    x = np.pi * diameter_mas * MAS_TO_RAD * sf
    return (2 * j1(x) / x) ** 2


first_lobe = vis2_sf < 40e6
(diameter,), cov = curve_fit(uniform_disk_v2, vis2_sf[first_lobe], vis2[first_lobe], p0=[3],
                             sigma=vis2_err[first_lobe])

distance_from_symmetric = np.minimum(np.abs(wrap(cp)), 180 - np.abs(wrap(cp)))
significance = distance_from_symmetric / cp_err
beyond_3 = np.mean(significance > 3)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4))
axes[0].errorbar(vis2_sf / 1e6, vis2, yerr=vis2_err, fmt="o", ms=2, color=BLUE, ecolor="#9ec5f4",
                 alpha=0.6)
curve = np.linspace(1e6, 230e6, 500)
axes[0].plot(curve / 1e6, uniform_disk_v2(curve, diameter), color=ORANGE,
             label=f"uniform disk {diameter:.2f} mas")
axes[0].set_yscale("log")
axes[0].set_ylim(1e-4, 1.2)
axes[0].set_xlabel("Spatial frequency (millions of wavelengths)")
axes[0].set_ylabel("V² (log scale)")
axes[0].set_title("CL Lac is heavily resolved", fontsize=10)
axes[0].legend(fontsize=8)

axes[1].scatter(cp_sf / 1e6, wrap(cp), s=3, color=BLUE, alpha=0.5)
for level in (-180, 0, 180):
    axes[1].axhline(level, color=ORANGE, lw=1)
axes[1].set_ylim(-190, 190)
axes[1].set_yticks([-180, -90, 0, 90, 180])
axes[1].set_xlabel("Longest baseline in the triangle (millions of wavelengths)")
axes[1].set_ylabel("Closure phase (°)")
axes[1].set_title("Closure phases (symmetric star: orange lines only)", fontsize=10)

edges = np.linspace(0, 10, 41)
axes[2].hist(np.minimum(significance, 9.99), bins=edges, density=True, color=BLUE, alpha=0.7,
             label="CL Lac")
x = np.linspace(0, 10, 200)
axes[2].plot(x, 2 * np.exp(-x ** 2 / 2) / np.sqrt(2 * np.pi), color=ORANGE,
             label="expected for a symmetric star")
axes[2].axvline(3, color=GREY, ls="--", lw=1)
axes[2].set_xlabel("Distance from 0° or 180°, in error bars (last bin: 10 or more)")
axes[2].set_ylabel("Fraction per bin")
axes[2].set_title(f"{100 * beyond_3:.0f}% are more than 3 error bars away", fontsize=10)
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Uniform disk diameter from the first lobe: {diameter:.2f} ± {np.sqrt(cov[0, 0]):.2f} mas")
print(f"Closure phases more than 3 error bars from 0° or 180°: {100 * beyond_3:.0f}% "
      f"(a symmetric star would give about 0.3%)")

What the real data show:

- Many closure phases sit near 0° or ±180°. At coarse resolution CL Lac looks like a round disk. Values near 180° appear where one baseline is past the first null, where a disk's visibility changes sign.
- But a large fraction sit far from both, by many error bars (right panel). For a symmetric star, random errors would put only about 0.3% beyond 3 error bars. Fine detail on the star's surface is **not** symmetric.

This is the clue that CL Lac has bright and dark patches, probably giant convection cells, which Chiavassa et al. imaged. Notebook 08 turns these $V^2$ and closure phases into a picture.

The uniform disk here is only a size estimate. Its fit is poor because a real red giant is limb-darkened and patchy. Chiavassa et al. fitted model atmospheres and found about 3.0–3.3 mas, larger than the uniform-disk value, as expected for a star dimmer at its edge.

## Summary

| Idea | What it means |
|---|---|
| Fried parameter $r_0$ | Size of a flat patch of wavefront. About 10 cm in visible light, growing as $\lambda^{6/5}$ |
| Speckles | A mirror much larger than $r_0$ sees a broken-up image |
| Piston | Random delay above each telescope, micrometres, changing in milliseconds |
| Short exposures | Frames of a few ms keep the contrast. $V^2$ is averaged because phases are random |
| Closure phase | Sum of phases around a triangle. Piston cancels. 0°/180° means symmetric |

1. The atmosphere scrambles each baseline's fringe phase, but not its contrast, if exposures are short.
2. Adding phases around a triangle cancels the atmosphere exactly.
3. Real MIRC-X closure phases show CL Lac's surface is not symmetric.

**Next:** T8, binaries and image reconstruction. Fitting binary stars, and turning $V^2$ and closure phases into images.